# ML4SCI GENIE GSoC 2026 — All Tasks Notebook

**Project:** Deep Graph Anomaly Detection with Contrastive Learning  
**Tasks:** Common Task 1 (CAE) · Common Task 2 (GAT Classifier) · Specific Task 1 (GraphCLR)

> ⚡ Runtime → Change runtime type → **T4 GPU** for ~10× speedup


## 1. Install Dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('torch==2.2.0', 'torchvision')

import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH}  cuda={CUDA}')

# PyG wheels
pip(
    f'torch_scatter torch_sparse torch_cluster torch_spline_conv '
    f'-f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'
)
pip('torch_geometric', 'h5py', 'pyarrow', 'scikit-learn', 'matplotlib', 'tqdm')

## 2. Download Dataset from Google Drive

In [ ]:
import os
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Official ML4SCI quark/gluon dataset (139 k events, ~700 MB)
FILE_ID = '1WO2K-SfU2dntGU4Bb3IYBp9Rh7rtTYEr'
OUTPUT  = 'data/quark-gluon_data-set_n139306.hdf5'

if not os.path.exists(OUTPUT):
    !pip install -q gdown
    import gdown
    gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', OUTPUT, quiet=False)
else:
    print('Dataset already downloaded.')

import h5py
with h5py.File(OUTPUT) as f:
    print('Keys:', list(f.keys()))
    print('X_jets shape:', f['X_jets'].shape)

## 3. Shared Utilities

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from tqdm.notebook import tqdm

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, global_max_pool
from torch_geometric.transforms import KNNGraph

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Data loading ──────────────────────────────────────────────
def load_dataset(path, max_events=None):
    with h5py.File(path) as f:
        sl = slice(None) if max_events is None else slice(max_events)
        X  = f['X_jets'][sl].astype(np.float32)  # (N,125,125,3)
        y  = f['y'][sl].astype(np.int64)
    X = np.transpose(X, (0, 3, 1, 2))             # → (N,3,125,125)
    print(f'Loaded {len(X):,} events | quark={np.sum(y==0):,} gluon={np.sum(y==1):,}')
    return X, y

def split_data(y, train_frac=0.70, val_frac=0.15, seed=42):
    n   = len(y)
    idx = np.arange(n)
    np.random.default_rng(seed).shuffle(idx)
    nt = int(n * train_frac)
    nv = int(n * val_frac)
    return idx[:nt], idx[nt:nt+nv], idx[nt+nv:]

# ── Image → point cloud ──────────────────────────────────────
def image_to_pointcloud(img, threshold=0.0):
    ch_track, ch_ecal, ch_hcal = img[0], img[1], img[2]
    mask = (np.abs(ch_track) + np.abs(ch_ecal) + np.abs(ch_hcal)) > threshold
    eta_idx, phi_idx = np.where(mask)
    if len(eta_idx) == 0:
        return np.zeros((1, 5), dtype=np.float32)
    eta_norm = (eta_idx / 124.0) * 2.0 - 1.0
    phi_norm = (phi_idx / 124.0) * 2.0 - 1.0
    points = np.column_stack([
        eta_norm, phi_norm,
        ch_track[eta_idx, phi_idx],
        ch_ecal [eta_idx, phi_idx],
        ch_hcal [eta_idx, phi_idx]
    ]).astype(np.float32)
    return points

print('Utilities ready.')

---
## Task 1 — Convolutional Autoencoder (CAE)

In [ ]:
MAX_EVENTS_T1 = 20_000
BATCH_SIZE_T1 = 64
EPOCHS_T1     = 30

X, y = load_dataset('data/quark-gluon_data-set_n139306.hdf5', max_events=MAX_EVENTS_T1)
max_val = X.max()
X_norm  = X / (max_val + 1e-8)   # normalise to [0,1]

class JetImageDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_idx, val_idx, test_idx = split_data(y)
train_ds = JetImageDataset(X_norm[train_idx], y[train_idx])
val_ds   = JetImageDataset(X_norm[val_idx],   y[val_idx])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_T1, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE_T1, shuffle=False, num_workers=2, pin_memory=True)

class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64,32,2,stride=2), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32,16,2,stride=2), nn.BatchNorm2d(16), nn.ReLU(True),
            nn.ConvTranspose2d(16,8,2,stride=2),  nn.BatchNorm2d(8),  nn.ReLU(True),
            nn.Upsample(size=(125,125), mode='bilinear', align_corners=False),
            nn.Conv2d(8,3,3,padding=1), nn.Sigmoid(),
        )
    def forward(self, x): return self.decoder(self.encoder(x))

model_cae  = ConvAutoencoder().to(DEVICE)
criterion  = nn.MSELoss()
opt_cae    = torch.optim.Adam(model_cae.parameters(), lr=1e-3)
sched_cae  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cae, T_max=EPOCHS_T1)

train_losses, val_losses = [], []
best_val, best_state = float('inf'), None

for ep in range(1, EPOCHS_T1+1):
    model_cae.train()
    tr_loss = 0.0
    for imgs, _ in tqdm(train_loader, leave=False, desc=f'Epoch {ep}/{EPOCHS_T1}'):
        imgs = imgs.to(DEVICE)
        loss = criterion(model_cae(imgs), imgs)
        opt_cae.zero_grad(); loss.backward(); opt_cae.step()
        tr_loss += loss.item() * imgs.size(0)
    tr_loss /= len(train_ds)

    model_cae.eval()
    vl_loss = 0.0
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(DEVICE)
            vl_loss += criterion(model_cae(imgs), imgs).item() * imgs.size(0)
    vl_loss /= len(val_ds)
    sched_cae.step()

    train_losses.append(tr_loss); val_losses.append(vl_loss)
    if vl_loss < best_val:
        best_val = vl_loss
        best_state = {k: v.clone() for k, v in model_cae.state_dict().items()}
    if ep % 5 == 0: print(f'Ep {ep:3d}  train={tr_loss:.5f}  val={vl_loss:.5f}')

model_cae.load_state_dict(best_state)
print(f'\nBest Val MSE: {best_val:.6f}')

In [ ]:
# ── Visualise reconstructions ──────────────────────────────────
CH_NAMES = ['Tracks', 'ECAL', 'HCAL']
N_SHOW   = 8
model_cae.eval()

imgs_t = torch.stack([val_ds[i][0] for i in range(N_SHOW)]).to(DEVICE)
with torch.no_grad():
    recons_t = model_cae(imgs_t).cpu().numpy()
imgs_np = imgs_t.cpu().numpy()

fig, axes = plt.subplots(N_SHOW, 6, figsize=(18, N_SHOW*2.2))
fig.suptitle('CAE Reconstruction — Quark/Gluon Jets', fontsize=14, fontweight='bold')
for row in range(N_SHOW):
    for ch in range(3):
        vmax = imgs_np[row, ch].max() + 1e-6
        axes[row, ch].imshow(imgs_np[row, ch], cmap='hot', vmin=0, vmax=vmax)
        axes[row, ch+3].imshow(recons_t[row, ch], cmap='hot', vmin=0, vmax=vmax)
        if row == 0:
            axes[row, ch].set_title(f'Orig {CH_NAMES[ch]}', fontsize=8)
            axes[row, ch+3].set_title(f'Recon {CH_NAMES[ch]}', fontsize=8)
        axes[row, ch].axis('off'); axes[row, ch+3].axis('off')
plt.tight_layout()
plt.savefig('outputs/task1_reconstructions.png', dpi=150, bbox_inches='tight')
plt.show()

fig2, ax2 = plt.subplots(figsize=(7,4))
ax2.plot(train_losses, label='Train MSE', color='#2563EB', lw=2)
ax2.plot(val_losses,   label='Val MSE',   color='#DC2626', lw=2, linestyle='--')
ax2.set(xlabel='Epoch', ylabel='MSE Loss', title='CAE Training Loss Curve')
ax2.legend(); ax2.grid(alpha=0.3)
plt.savefig('outputs/task1_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task 2 — GNN Jet Classifier (Graph Attention Network)

In [ ]:
MAX_EVENTS_T2 = 20_000
KNN_K         = 8
BATCH_GNN     = 32
EPOCHS_GNN    = 30

X2, y2 = load_dataset('data/quark-gluon_data-set_n139306.hdf5', max_events=MAX_EVENTS_T2)
train_idx2, val_idx2, test_idx2 = split_data(y2)

knn_tf = KNNGraph(k=KNN_K, loop=False, force_undirected=True)

def img_to_graph(img, label):
    pts = image_to_pointcloud(img)
    x   = torch.tensor(pts, dtype=torch.float)
    d   = Data(x=x, pos=x[:, :2], y=torch.tensor([label], dtype=torch.long))
    return knn_tf(d)

class JetGraphDS(Dataset):
    def __init__(self, X, y): self.X=X; self.y=y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return img_to_graph(self.X[i], int(self.y[i]))

collate = lambda b: Batch.from_data_list(b)
train_gnn = DataLoader(JetGraphDS(X2[train_idx2],y2[train_idx2]), batch_size=BATCH_GNN, shuffle=True,  collate_fn=collate, num_workers=2)
val_gnn   = DataLoader(JetGraphDS(X2[val_idx2],  y2[val_idx2]),   batch_size=BATCH_GNN, shuffle=False, collate_fn=collate, num_workers=2)
test_gnn  = DataLoader(JetGraphDS(X2[test_idx2], y2[test_idx2]),  batch_size=BATCH_GNN, shuffle=False, collate_fn=collate, num_workers=2)

class JetGAT(nn.Module):
    def __init__(self, in_ch=5):
        super().__init__()
        self.conv1 = GATConv(in_ch, 32, heads=4, concat=True, dropout=0.1)
        self.bn1   = nn.BatchNorm1d(128)
        self.conv2 = GATConv(128, 32, heads=4, concat=True, dropout=0.1)
        self.bn2   = nn.BatchNorm1d(128)
        self.conv3 = GATConv(128, 64, heads=1, concat=False, dropout=0.1)
        self.clf   = nn.Sequential(
            nn.Linear(128, 64), nn.ELU(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, x, ei, batch):
        h = F.elu(self.bn1(self.conv1(x, ei)))
        h = F.elu(self.bn2(self.conv2(h, ei)))
        h = F.elu(self.conv3(h, ei))
        g = torch.cat([global_mean_pool(h,batch), global_max_pool(h,batch)], 1)
        return self.clf(g).squeeze(-1)

model_gat = JetGAT().to(DEVICE)
pos_w     = torch.tensor([y2[train_idx2].mean()/(1-y2[train_idx2].mean()+1e-6)], device=DEVICE)
crit_gat  = nn.BCEWithLogitsLoss(pos_weight=pos_w)
opt_gat   = torch.optim.AdamW(model_gat.parameters(), lr=1e-3, weight_decay=1e-4)
sched_gat = torch.optim.lr_scheduler.CosineAnnealingLR(opt_gat, T_max=EPOCHS_GNN, eta_min=1e-5)

best_auc_gat, best_state_gat = 0.0, None
for ep in range(1, EPOCHS_GNN+1):
    model_gat.train()
    tr_loss = 0.0
    for b in tqdm(train_gnn, leave=False, desc=f'GAT Ep {ep}'):
        b = b.to(DEVICE)
        l = crit_gat(model_gat(b.x,b.edge_index,b.batch), b.y.float())
        opt_gat.zero_grad(); l.backward()
        torch.nn.utils.clip_grad_norm_(model_gat.parameters(), 1.0)
        opt_gat.step(); tr_loss += l.item()*b.num_graphs
    tr_loss /= len(train_gnn.dataset)

    model_gat.eval(); vl_logits, vl_y = [], []
    with torch.no_grad():
        for b in val_gnn:
            b=b.to(DEVICE); vl_logits.extend(model_gat(b.x,b.edge_index,b.batch).cpu()); vl_y.extend(b.y.cpu())
    vl_probs = torch.sigmoid(torch.tensor(vl_logits)).numpy()
    vl_auc   = roc_auc_score(np.array(vl_y), vl_probs)
    sched_gat.step()
    if vl_auc > best_auc_gat:
        best_auc_gat  = vl_auc
        best_state_gat= {k:v.clone() for k,v in model_gat.state_dict().items()}
    if ep%5==0: print(f'Ep {ep:3d}  train={tr_loss:.4f}  val_auc={vl_auc:.4f}  best={best_auc_gat:.4f}')

model_gat.load_state_dict(best_state_gat)
model_gat.eval(); te_logits, te_y = [], []
with torch.no_grad():
    for b in test_gnn:
        b=b.to(DEVICE); te_logits.extend(model_gat(b.x,b.edge_index,b.batch).cpu()); te_y.extend(b.y.cpu())
te_probs = torch.sigmoid(torch.tensor(te_logits)).numpy()
te_y     = np.array(te_y)
te_auc   = roc_auc_score(te_y, te_probs)
te_acc   = accuracy_score(te_y, (te_probs>0.5).astype(int))
print(f'\n[Task 2] Test Accuracy={te_acc:.4f}  Test AUC={te_auc:.4f}')

In [ ]:
fpr, tpr, _ = roc_curve(te_y, te_probs)
fig, ax = plt.subplots(figsize=(6,6))
ax.plot(fpr, tpr, color='#7C3AED', lw=2.5, label=f'GAT  AUC={te_auc:.4f}')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color='#7C3AED')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
       title='Task 2 — GAT Jet Classifier ROC Curve')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
ax.text(0.55, 0.15, f'AUC = {te_auc:.4f}', transform=ax.transAxes, fontsize=13,
        color='#7C3AED', fontweight='bold')
plt.savefig('outputs/task2_roc.png', dpi=150, bbox_inches='tight'); plt.show()

---
## Task 3 — Contrastive Anomaly Detection (GraphCLR)

In [ ]:
MAX_EVENTS_T3 = 20_000
TEMP     = 0.5
EMB_DIM  = 128
GNN_DIM  = 128
EP_CLR   = 50
EP_PROBE = 20
BATCH_CLR= 64

X3, y3 = load_dataset('data/quark-gluon_data-set_n139306.hdf5', max_events=MAX_EVENTS_T3)
train_idx3, val_idx3, test_idx3 = split_data(y3)

# ── Augmentations ─────────────────────────────────────────────
def augment(data, ndrop=0.1, edrop=0.2, noise=0.05):
    x, ei, n = data.x.clone(), data.edge_index.clone(), data.x.size(0)
    keep = torch.rand(n) > ndrop
    keep[:2] = True
    kn   = keep.nonzero(as_tuple=True)[0]
    remap = torch.full((n,), -1); remap[kn] = torch.arange(kn.size(0))
    src, dst = ei; ve = keep[src] & keep[dst]
    ei = torch.stack([remap[src[ve]], remap[dst[ve]]])
    x  = x[kn]
    if ei.size(1)>0:
        ke = torch.rand(ei.size(1)) > edrop; ke[0]=True; ei=ei[:,ke]
    return Data(x=x+torch.randn_like(x)*noise, edge_index=ei)

class CLRDataset(Dataset):
    def __init__(self, X, y): self.X=X; self.y=y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        pts = image_to_pointcloud(self.X[i])
        x   = torch.tensor(pts, dtype=torch.float)
        d   = Data(x=x, pos=x[:,:2]); d=knn_tf(d)
        return augment(d), augment(d), int(self.y[i])

def coll_clr(batch):
    v1,v2,lb = zip(*batch)
    return Batch.from_data_list(v1), Batch.from_data_list(v2), torch.tensor(lb)

tr3 = DataLoader(CLRDataset(X3[train_idx3],y3[train_idx3]),BATCH_CLR,True, collate_fn=coll_clr,num_workers=2)
va3 = DataLoader(CLRDataset(X3[val_idx3],  y3[val_idx3]),  BATCH_CLR,False,collate_fn=coll_clr,num_workers=2)
te3 = DataLoader(CLRDataset(X3[test_idx3], y3[test_idx3]), BATCH_CLR,False,collate_fn=coll_clr,num_workers=2)

# ── GAT Encoder ───────────────────────────────────────────────
class GATEnc(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1=GATConv(5,  32,heads=4,concat=True, dropout=0.1); self.b1=nn.BatchNorm1d(128)
        self.c2=GATConv(128,32,heads=4,concat=True, dropout=0.1); self.b2=nn.BatchNorm1d(128)
        self.c3=GATConv(128,64,heads=1,concat=False,dropout=0.1)
    def forward(self, x, ei, batch):
        h=F.elu(self.b1(self.c1(x,ei))); h=F.elu(self.b2(self.c2(h,ei))); h=F.elu(self.c3(h,ei))
        return torch.cat([global_mean_pool(h,batch), global_max_pool(h,batch)],1)

class GraphCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc  = GATEnc()
        self.proj = nn.Sequential(
            nn.Linear(GNN_DIM,GNN_DIM), nn.BatchNorm1d(GNN_DIM), nn.ReLU(True),
            nn.Linear(GNN_DIM,EMB_DIM))
    def encode(self,x,ei,b): return self.enc(x,ei,b)
    def forward(self,x,ei,b):
        h=self.enc(x,ei,b); z=self.proj(h); return F.normalize(z,dim=1)

def nt_xent(zi, zj, tau=TEMP):
    N=zi.size(0); z=torch.cat([zi,zj])
    sim=torch.mm(z,z.T)/tau
    sim.masked_fill_(torch.eye(2*N,dtype=bool,device=z.device), float('-inf'))
    return F.cross_entropy(sim, torch.cat([torch.arange(N,2*N), torch.arange(N)]).to(z.device))

model_clr = GraphCLR().to(DEVICE)
opt_clr   = torch.optim.Adam(model_clr.parameters(), lr=3e-4, weight_decay=1e-5)
sch_clr   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_clr, T_max=EP_CLR, eta_min=1e-5)

print(f'GraphCLR params: {sum(p.numel() for p in model_clr.parameters()):,}')
best_clr = float('inf'); best_clr_st = None
for ep in range(1, EP_CLR+1):
    model_clr.train(); tot=0.0
    for v1,v2,_ in tqdm(tr3, leave=False, desc=f'CLR Ep {ep}'):
        v1,v2=v1.to(DEVICE),v2.to(DEVICE)
        loss=nt_xent(model_clr(v1.x,v1.edge_index,v1.batch),
                     model_clr(v2.x,v2.edge_index,v2.batch))
        opt_clr.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model_clr.parameters(),1.0); opt_clr.step()
        tot+=loss.item()*v1.num_graphs
    tot/=len(tr3.dataset); sch_clr.step()
    if tot<best_clr: best_clr=tot; best_clr_st={k:v.clone() for k,v in model_clr.state_dict().items()}
    if ep%5==0: print(f'Ep {ep:3d}  NT-Xent={tot:.4f}  best={best_clr:.4f}')

model_clr.load_state_dict(best_clr_st)

In [ ]:
# ── Anomaly scoring ────────────────────────────────────────────
model_clr.eval()
def get_embeds(loader):
    hs, ys = [], []
    with torch.no_grad():
        for v1,v2,lb in loader:
            v1=v1.to(DEVICE); hs.append(model_clr.encode(v1.x,v1.edge_index,v1.batch).cpu()); ys.append(lb)
    return torch.cat(hs).numpy(), torch.cat(ys).numpy()

trH, trY = get_embeds(tr3)
teH, teY = get_embeds(te3)
centroid  = trH.mean(0)
scores    = np.linalg.norm(teH - centroid, axis=1)
auc_unsup = roc_auc_score(teY, scores)
print(f'Unsupervised anomaly AUC: {auc_unsup:.4f}')

# ── Linear probe ──────────────────────────────────────────────
vaH, vaY = get_embeds(va3)
for p in model_clr.enc.parameters(): p.requires_grad=False
probe   = nn.Linear(GNN_DIM, 2).to(DEVICE)
opt_pr  = torch.optim.Adam(probe.parameters(), lr=1e-3)
trHt, trYt = torch.tensor(trH), torch.tensor(trY)
for _ in range(EP_PROBE):
    probe.train(); logits=probe(trHt.to(DEVICE))
    loss=F.cross_entropy(logits, trYt.to(DEVICE))
    opt_pr.zero_grad(); loss.backward(); opt_pr.step()
probe.eval()
with torch.no_grad(): te_prob=F.softmax(probe(torch.tensor(teH).to(DEVICE)),1)[:,1].cpu().numpy()
auc_sup = roc_auc_score(teY, te_prob)
acc_sup = accuracy_score(teY, (te_prob>0.5).astype(int))
print(f'Linear-probe AUC={auc_sup:.4f}  Accuracy={acc_sup:.4f}')

In [ ]:
def plot_roc(y, s, auc, title, path, color='#DC2626'):
    fpr, tpr, _ = roc_curve(y, s)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'AUC = {auc:.4f}')
    ax.plot([0,1],[0,1],'k--',lw=1); ax.fill_between(fpr,tpr,alpha=0.08,color=color)
    ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title=title)
    ax.legend(loc='lower right'); ax.grid(alpha=0.3)
    ax.text(0.55,0.15,f'AUC = {auc:.4f}',transform=ax.transAxes,
            fontsize=13,color=color,fontweight='bold')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show()

plot_roc(teY, scores, auc_unsup, 'Task 3 — Contrastive Anomaly Detection',
         'outputs/task3_roc_anomaly.png', '#DC2626')
plot_roc(teY, te_prob, auc_sup, 'Task 3 — Contrastive + Linear Probe Classification',
         'outputs/task3_roc.png', '#059669')

---
## Summary of Results

In [ ]:
print('='*55)
print('  ML4SCI GENIE GSoC 2026 — Results Summary')
print('='*55)
print(f'  Task 1 — CAE         Val MSE          : {best_val:.6f}')
print(f'  Task 2 — GAT         Test ROC-AUC     : {te_auc:.4f}')
print(f'  Task 2 — GAT         Test Accuracy    : {te_acc:.4f}')
print(f'  Task 3 — GraphCLR    Anomaly AUC      : {auc_unsup:.4f}')
print(f'  Task 3 — LinearProbe Classification AUC: {auc_sup:.4f}')
print('='*55)